# 🛣️ SafeRoad AI — YOLOv8 Road Hazard Detector
### Kaggle Training Notebook — No Dataset Download Required!

---

## ✅ Setup Instructions (Do these BEFORE running):

### 1. Enable GPU:
> **Settings** (right panel) → **Accelerator** → Select **GPU T4 x2** → Save

### 2. Add Dataset:
> **+ Add Data** (right panel) → Search **"pothole detection yolov8"** → Click **Add**
>
> ✅ Recommended datasets to add:
> - `atulyakumar98/pothole-detection-dataset`
> - `andrewmvd/pothole-detection` (has YOLO format)

### 3. Run All Cells:
> **Run All** ▶️ → Wait ~15–30 mins → Download `best.pt` from Output tab

---

## ⚙️ Step 1 — Check GPU & Environment

In [ ]:
import torch
import os

print("=" * 50)
print("  🛣️  SafeRoad AI — Kaggle Training Environment")
print("=" * 50)
print(f"  PyTorch  : {torch.__version__}")
print(f"  CUDA     : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"  GPU      : {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  VRAM     : {vram:.1f} GB")
    print("  ✅ GPU is ready!")
else:
    print("  ⚠️  No GPU found!")
    print("  → Go to Settings → Accelerator → GPU T4 x2")

# Show available input datasets
print("\n📂 Available Input Datasets:")
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    datasets = os.listdir(input_dir)
    if datasets:
        for d in datasets:
            print(f"   ✅ /kaggle/input/{d}")
    else:
        print("   ⚠️  No datasets added yet!")
        print("   → Click '+ Add Data' on the right panel")
else:
    print("   ⚠️  /kaggle/input not found")

## 📦 Step 2 — Install YOLOv8

In [ ]:
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f"✅ Ultralytics YOLOv8 version: {ultralytics.__version__}")

## 📂 Step 3 — Auto-Detect Dataset

In [ ]:
import glob
import yaml
import os

# ── Auto-find data.yaml from any added Kaggle dataset ──────────────────────
yaml_files = glob.glob("/kaggle/input/**/*.yaml", recursive=True)
yaml_files += glob.glob("/kaggle/input/**/*.yml", recursive=True)

# Filter to YOLO-style yamls (must have 'train' and 'names' keys)
valid_yamls = []
for yf in yaml_files:
    try:
        with open(yf) as f:
            content = yaml.safe_load(f)
        if isinstance(content, dict) and 'names' in content:
            valid_yamls.append(yf)
    except:
        pass

print("🔍 Found YOLO data.yaml files:")
for i, y in enumerate(valid_yamls):
    print(f"   [{i}] {y}")

if valid_yamls:
    DATA_YAML = valid_yamls[0]
    print(f"\n✅ Using: {DATA_YAML}")
    with open(DATA_YAML) as f:
        cfg = yaml.safe_load(f)
    print("\n📋 Dataset config:")
    print(f"   Classes : {cfg.get('names', {})}")
    nc = len(cfg.get('names', {})) if isinstance(cfg.get('names'), dict) else len(cfg.get('names', []))
    print(f"   Num classes: {nc}")
else:
    print("\n⚠️  No YOLO yaml found! Using synthetic dataset fallback...")
    DATA_YAML = None

## 🔧 Step 4 — Fix Dataset Paths for Kaggle

> Kaggle datasets are read-only at `/kaggle/input/`. This cell fixes any absolute paths in `data.yaml` so YOLOv8 can find them.

In [ ]:
import yaml
import os
from pathlib import Path

if DATA_YAML:
    with open(DATA_YAML) as f:
        cfg = yaml.safe_load(f)

    dataset_root = str(Path(DATA_YAML).parent)

    # Fix the 'path' key to point to Kaggle input directory
    cfg['path'] = dataset_root

    # Check if train/val paths exist, fix if needed
    for key in ['train', 'val', 'test']:
        if key in cfg:
            candidate = os.path.join(dataset_root, cfg[key])
            if os.path.exists(candidate):
                print(f"  ✅ {key}: {candidate}")
            else:
                # Try to find the split folder
                found = glob.glob(f"{dataset_root}/**/{key}", recursive=True)
                if found:
                    cfg[key] = os.path.relpath(found[0], dataset_root)
                    print(f"  🔧 {key} fixed → {cfg[key]}")
                else:
                    print(f"  ⚠️  {key} path not found: {candidate}")

    # Write fixed yaml to /kaggle/working (writable)
    FIXED_YAML = "/kaggle/working/data_fixed.yaml"
    with open(FIXED_YAML, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False)

    DATA_YAML = FIXED_YAML
    print(f"\n✅ Fixed data.yaml saved to: {DATA_YAML}")
    print("\n📄 Final config:")
    with open(DATA_YAML) as f:
        print(f.read())
else:
    print("⏭️ Skipping path fix (will use synthetic dataset)")

## 🔁 Step 4B — Fallback: Generate Synthetic Dataset

> **Only runs if no dataset was added.** Skip this if Step 3 found a valid dataset.

In [ ]:
if DATA_YAML is None:
    print("⚡ No dataset added — generating synthetic road dataset...")
    import numpy as np
    import cv2
    import random
    from pathlib import Path

    DSET = Path("/kaggle/working/synthetic_road")
    NUM_TRAIN, NUM_VAL, SZ = 400, 100, 640

    for split in ["train", "val"]:
        (DSET/"images"/split).mkdir(parents=True, exist_ok=True)
        (DSET/"labels"/split).mkdir(parents=True, exist_ok=True)

    def asphalt(w, h):
        img = np.full((h,w,3), random.randint(60,110), dtype=np.uint8)
        img = np.clip(img.astype(np.int16)+np.random.normal(0,14,(h,w,3)).astype(np.int16),0,255).astype(np.uint8)
        if random.random()<0.6:
            lx=random.randint(int(w*.1),int(w*.9))
            cv2.line(img,(lx,0),(lx+random.randint(-30,30),h),(220,220,220),random.randint(6,14))
        return np.clip(img*np.linspace(0.75,1.15,h)[:,None,None],0,255).astype(np.uint8)

    def pothole(img, L):
        h,w=img.shape[:2]
        cx,cy=random.randint(int(w*.2),int(w*.8)),random.randint(int(h*.2),int(h*.8))
        rx=random.randint(int(w*.05),int(w*.15)); ry=int(rx*random.uniform(.45,.85))
        cv2.ellipse(img,(cx,cy),(rx,ry),random.randint(0,180),0,360,(20,20,20),-1)
        cv2.ellipse(img,(cx,cy),(int(rx*.75),int(ry*.75)),0,0,360,(10,10,10),-1)
        bx1,by1,bx2,by2=max(0,cx-rx-4),max(0,cy-ry-4),min(w,cx+rx+4),min(h,cy+ry+4)
        L.append(f"0 {((bx1+bx2)/2)/w:.6f} {((by1+by2)/2)/h:.6f} {(bx2-bx1)/w:.6f} {(by2-by1)/h:.6f}")

    def crack(img, L):
        h,w=img.shape[:2]
        cx,cy=random.randint(int(w*.25),int(w*.75)),random.randint(int(h*.25),int(h*.75))
        rx=random.randint(int(w*.08),int(w*.18)); ry=int(rx*random.uniform(.6,1.0))
        pts=[(np.clip(int(cx+random.gauss(0,rx*.45)),10,w-10),np.clip(int(cy+random.gauss(0,ry*.45)),10,h-10)) for _ in range(20)]
        for i in range(len(pts)):
            for j in range(i+1,min(i+4,len(pts))):
                if random.random()<.7: cv2.line(img,pts[i],pts[j],(15,15,15),random.randint(1,3))
        bx1=max(0,min(p[0] for p in pts)-6); by1=max(0,min(p[1] for p in pts)-6)
        bx2=min(w,max(p[0] for p in pts)+6); by2=min(h,max(p[1] for p in pts)+6)
        L.append(f"1 {((bx1+bx2)/2)/w:.6f} {((by1+by2)/2)/h:.6f} {(bx2-bx1)/w:.6f} {(by2-by1)/h:.6f}")

    for split,count in [("train",NUM_TRAIN),("val",NUM_VAL)]:
        for i in range(count):
            img=asphalt(SZ,SZ); L=[]
            if random.random()<.85:
                for _ in range(random.choices([1,2,3],weights=[.6,.3,.1])[0]):
                    (pothole if random.random()<.6 else crack)(img,L)
            cv2.imwrite(str(DSET/"images"/split/f"r_{i:04d}.jpg"),img,[cv2.IMWRITE_JPEG_QUALITY,93])
            with open(DSET/"labels"/split/f"r_{i:04d}.txt","w") as f: f.write("\n".join(L)+("\n" if L else ""))
        print(f"  ✅ {split}: {count} images generated")

    DATA_YAML = str(DSET/"data.yaml")
    with open(DATA_YAML,"w") as f:
        f.write(f"path: {DSET}\ntrain: images/train\nval: images/val\nnames:\n  0: Pothole\n  1: Alligator Crack\n  2: Longitudinal Crack\n  3: Severe Edge Erosion\n")
    print(f"\n✅ Synthetic dataset ready! data.yaml → {DATA_YAML}")
else:
    print("✅ Real dataset found — skipping synthetic generation.")

## 🚀 Step 5 — Train YOLOv8

> ⚙️ Tune `BASE_MODEL` and `EPOCHS` below before running.

In [ ]:
from ultralytics import YOLO
import torch

# ══════════════════════════════════════════════
#   ⚙️  CHANGE THESE TO TUNE YOUR MODEL
# ══════════════════════════════════════════════
BASE_MODEL = "yolov8s.pt"  # yolov8n=fastest | yolov8s=balanced✅ | yolov8m=accurate | yolov8l=best
EPOCHS     = 50            # 50 for real data | 30 for synthetic
BATCH      = 16            # Reduce to 8 if CUDA out-of-memory
IMG_SIZE   = 640           # 640 standard | 1280 higher accuracy
PATIENCE   = 15            # Early stopping patience
# ══════════════════════════════════════════════

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 55)
print("  🚀 SafeRoad AI — Starting YOLOv8 Training")
print("=" * 55)
print(f"  Base Model : {BASE_MODEL}")
print(f"  Dataset    : {DATA_YAML}")
print(f"  Device     : {device.upper()}")
print(f"  Epochs     : {EPOCHS}")
print(f"  Batch      : {BATCH}")
print(f"  Image Size : {IMG_SIZE}x{IMG_SIZE}")
print("=" * 55)

model = YOLO(BASE_MODEL)

results = model.train(
    data       = DATA_YAML,
    epochs     = EPOCHS,
    imgsz      = IMG_SIZE,
    batch      = BATCH,
    device     = device,
    project    = "/kaggle/working/saferoad_runs",
    name       = "road_hazard_yolov8",
    patience   = PATIENCE,
    save       = True,
    # ── Optimizer ──────────────────────────────
    optimizer  = "AdamW",
    lr0        = 0.001,
    lrf        = 0.01,
    # ── Road-specific augmentations ────────────
    mosaic     = 1.0,    # 4-image mosaic — great for small cracks
    mixup      = 0.15,   # reduces false positives
    copy_paste = 0.1,    # copies objects between images
    hsv_h      = 0.015,  # hue — different asphalt types
    hsv_s      = 0.6,    # saturation — wet vs dry
    hsv_v      = 0.4,    # brightness — shadows / tunnels
    degrees    = 10.0,   # rotation — dashcam tilt
    translate  = 0.1,    # translation — road bumps
    scale      = 0.5,    # scale variation
    fliplr     = 0.5,    # horizontal flip
    erasing    = 0.4,    # occlusion robustness
    verbose    = True
)

print("\n✅ Training complete!")

## 📊 Step 6 — View Results & Metrics

In [ ]:
from IPython.display import Image, display
import glob

# Print final metrics
metrics = model.val()

print("\n" + "=" * 55)
print("  📊 SafeRoad AI — Final Model Performance")
print("=" * 55)
print(f"  mAP@50      : {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)")
print(f"  mAP@50-95   : {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)")
print(f"  Precision   : {metrics.box.mp:.4f}")
print(f"  Recall      : {metrics.box.mr:.4f}")
print("=" * 55)

# Show training charts
run_dir = "/kaggle/working/saferoad_runs/road_hazard_yolov8"
charts = sorted(glob.glob(f"{run_dir}/*.png"))
for chart in charts[:6]:
    print(f"\n📈 {os.path.basename(chart)}")
    display(Image(chart, width=700))

## 🧪 Step 7 — Test Inference

In [ ]:
import numpy as np
import cv2
import time
from IPython.display import Image, display

# Load best weights
BEST_PT = "/kaggle/working/saferoad_runs/road_hazard_yolov8/weights/best.pt"
best_model = YOLO(BEST_PT)

# ── Option A: Test on a real image from the dataset ────────────────────────
test_images = glob.glob("/kaggle/input/**/images/**/*.jpg", recursive=True)
if test_images:
    test_img_path = test_images[0]
    print(f"🖼️  Testing on: {test_img_path}")
else:
    # ── Option B: Synthesize test image ───────────────────────────────────
    w, h = 640, 360
    img = np.full((h,w,3), 85, dtype=np.uint8)
    img = np.clip(img.astype(np.int16)+np.random.normal(0,12,(h,w,3)).astype(np.int16),0,255).astype(np.uint8)
    cv2.line(img,(w//2,0),(w//2,h),(220,220,220),8)
    cv2.ellipse(img,(180,240),(60,35),0,0,360,(20,20,20),-1)
    cv2.ellipse(img,(180,240),(45,22),0,0,360,(10,10,10),-1)
    cv2.line(img,(420,100),(435,280),(18,18,18),3)
    test_img_path = "/kaggle/working/test_frame.jpg"
    cv2.imwrite(test_img_path, img)
    print(f"🖼️  Using synthetic test image: {test_img_path}")

# Run inference
t0 = time.perf_counter()
preds = best_model.predict(test_img_path, conf=0.25, verbose=False)
ms = (time.perf_counter()-t0)*1000

boxes = preds[0].boxes
print(f"\n⚡ Speed   : {ms:.1f} ms  ({1000/max(ms,1):.0f} FPS)")
print(f"🔍 Detections: {len(boxes)}")
for i, b in enumerate(boxes):
    name = best_model.names[int(b.cls[0])]
    conf = float(b.conf[0]) * 100
    print(f"   [{i+1}] {name} — {conf:.1f}% confidence")

# Display annotated result
out = preds[0].plot()
out_path = "/kaggle/working/detection_result.jpg"
cv2.imwrite(out_path, out)
print("\n🖼️ Result:")
display(Image(out_path, width=700))

## 💾 Step 8 — Save Model to Output

> After this cell, go to the **Output tab** on the right → Download `pothole_yolov8.pt`

In [ ]:
import shutil
import os

BEST_PT   = "/kaggle/working/saferoad_runs/road_hazard_yolov8/weights/best.pt"
OUTPUT_PT = "/kaggle/working/pothole_yolov8.pt"

shutil.copy(BEST_PT, OUTPUT_PT)
size_mb = os.path.getsize(OUTPUT_PT) / 1e6

print("=" * 55)
print("  💾 SafeRoad AI — Model Saved!")
print("=" * 55)
print(f"  File : pothole_yolov8.pt")
print(f"  Size : {size_mb:.2f} MB")
print()
print("  📥 HOW TO DOWNLOAD:")
print("  → Look at the right panel → Output tab")
print("  → Find 'pothole_yolov8.pt'")
print("  → Click the download icon ⬇️")
print()
print("  📁 THEN PLACE IT IN YOUR PROJECT:")
print("  new road safe/server/models/pothole_yolov8.pt")
print("=" * 55)

## 📦 Step 9 — Export to ONNX (Optional)

In [ ]:
# Export to ONNX for web/mobile deployment
export_model = YOLO("/kaggle/working/saferoad_runs/road_hazard_yolov8/weights/best.pt")
onnx_path = export_model.export(format="onnx", dynamic=True, simplify=True)

import shutil
shutil.copy(onnx_path, "/kaggle/working/pothole_yolov8.onnx")
print(f"✅ ONNX exported → /kaggle/working/pothole_yolov8.onnx")
print("   Download from Output tab → place in server/models/")

---
## ✅ After Downloading

Place the file in your project:
```
new road safe/
  server/
    models/
      pothole_yolov8.pt   ← downloaded from Kaggle Output tab
```

Then start your local detection server:
```bash
python server/ml_server.py
```

---
**🛣️ SafeRoad AI** — Making roads safer with AI 🚗